In [1]:
%%bash
set -uo pipefail
# -u treat unset variables as errors
# -x print each command before executing (for debug only)
# -o fail if any comand in the pipeline fails
############################################
# USER SETTINGS

# DISCLAIMER: this code was written to run on
#TO RUN, YOU NEED TO RUN ALL CELLS, because otherwise ther will be no sound after finishing

#Fasta files input
R1="/home/jokubas/project/data/Teaching_Data/poll_1.fastq"
R2="/home/jokubas/project/data/Teaching_Data/poll_2.fastq"

#Define which threads we test
THREAD_LIST=(2 4)
#Rename sample name
SAMPLE="POLL"
#Output directory
OUTDIR="${SAMPLE}_resource_usage"
#Number of replicates
N_REPS=2

#Input the adapters manually
ADAPTER_R1="AGATCGGAAGAGCACACGTCTGAACTCCAGTCA"
ADAPTER_R2="AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT"

# Increase Java heap for Trimmomatic
# Before it was bootleneck by too little ram allocation
export _JAVA_OPTIONS="-Xms1g -Xmx8g"

############################################
# CREATE OUTPUT DIRECTORY
mkdir -p "$OUTDIR"

############################################
# RESOURCE LOG
RESOURCE_LOG="${OUTDIR}/${SAMPLE}_resource_usage.tsv"
#Make header
printf "threads\ttool\tcpu_percent\tcpu_total_seconds\tmax_rss_kb\telapsed_wall\tdisk_delta_bytes\n" > "$RESOURCE_LOG"

############################################
# FOR SIZE CALCULATION
dir_size_bytes() {
  du -sb "$1" 2>/dev/null | awk '{print $1}'
}
# Used to measure how much disk space the tools used later on
# This was not used in the data analysis, but still left here

############################################
# RUN TOOL AND LOG RESOURCE USAGE
# Building the resource log table and how we log the resources
run_and_log() {
  local threads="$1"
  local tool_name="$2"
  shift 2
  #Create temp file
  local tmp_time
  tmp_time="$(mktemp)"
  # Disk usage
  local before_bytes after_bytes delta_bytes
  # Resource variables
  local elapsed_wall cpu_percent max_rss_kb
  # CPU time usage
  local user_seconds system_seconds cpu_total_seconds
  # Measure directory size for disk usage calculations
  before_bytes="$(dir_size_bytes "$OUTDIR")"
  #Start message
  echo "Starting ${tool_name} (${threads} threads)"
  # This is the key function for logging:
  /usr/bin/time -v -o "$tmp_time" "$@"
  # Measure directory size after
  after_bytes="$(dir_size_bytes "$OUTDIR")"
  # Calculate the cahnge in size
  delta_bytes=$((after_bytes - before_bytes))

  elapsed_wall="$(grep -F "Elapsed (wall clock) time" "$tmp_time" | sed 's/.*: //')"
  cpu_percent="$(grep -F "Percent of CPU this job got" "$tmp_time" | sed 's/.*: //' | tr -d '%')"
  max_rss_kb="$(grep -F "Maximum resident set size" "$tmp_time" | sed 's/.*: //')"

  user_seconds="$(grep -F "User time (seconds)" "$tmp_time" | sed 's/.*: //')"
  system_seconds="$(grep -F "System time (seconds)" "$tmp_time" | sed 's/.*: //')"

  cpu_total_seconds="$(awk -v u="$user_seconds" -v s="$system_seconds" 'BEGIN {print u+s}')"

  printf "%s\t%s\t%s\t%s\t%s\t%s\t%s\n" \
    "$threads" \
    "$tool_name" \
    "$cpu_percent" \
    "$cpu_total_seconds" \
    "$max_rss_kb" \
    "$elapsed_wall" \
    "$delta_bytes" \
    >> "$RESOURCE_LOG"

  rm -f "$tmp_time"

}

#FOR LOOP
# First is for the replicates
# Second is for threads.
for REP in $(seq 1 "$N_REPS"); do
for THREADS in "${THREAD_LIST[@]}"; do

SAMPLE="COW_rep${REP}_threads_${THREADS}_"

############################################
# BUILD ADAPTER FASTA FOR TRIMMOMATIC

ADAPTER_FASTA="${OUTDIR}/${SAMPLE}_adapters.fa"

cat > "$ADAPTER_FASTA" <<EOF
>PrefixPE/1
${ADAPTER_R1}
>PrefixPE/2
${ADAPTER_R2}
EOF

############################################
# fastp

run_and_log "$THREADS" "fastp" \
  fastp \
    --in1 "$R1" \
    --in2 "$R2" \
    --out1 "${OUTDIR}/${SAMPLE}1_fastp.fastq" \
    --out2 "${OUTDIR}/${SAMPLE}2_fastp.fastq" \
    --adapter_sequence "$ADAPTER_R1" \
    --adapter_sequence_r2 "$ADAPTER_R2" \
    --cut_tail \
    --cut_tail_window_size 1 \
    --cut_tail_mean_quality 30 \
    --length_required 50 \
    --thread "$THREADS"

############################################
# cutadapt

run_and_log "$THREADS" "cutadapt" \
  cutadapt \
    -j "$THREADS" \
    -q 30 \
    -m 50 \
    -a "$ADAPTER_R1" \
    -A "$ADAPTER_R2" \
    -o "${OUTDIR}/${SAMPLE}1_cutadapt.fastq" \
    -p "${OUTDIR}/${SAMPLE}2_cutadapt.fastq" \
    "$R1" "$R2"

############################################
# trimmomatic

run_and_log "$THREADS" "trimmomatic" \
  trimmomatic PE \
    -threads "$THREADS" \
    -phred33 \
    "$R1" "$R2" \
    "${OUTDIR}/${SAMPLE}1_trimmomatic.fastq" \
    "${OUTDIR}/${SAMPLE}1_trimmomatic_unpaired.fastq" \
    "${OUTDIR}/${SAMPLE}2_trimmomatic.fastq" \
    "${OUTDIR}/${SAMPLE}2_trimmomatic_unpaired.fastq" \
    ILLUMINACLIP:"$ADAPTER_FASTA":2:30:10 \
    TRAILING:30 \
    MINLEN:50

rm -f \
  "${OUTDIR}/${SAMPLE}1_trimmomatic_unpaired.fastq" \
  "${OUTDIR}/${SAMPLE}2_trimmomatic_unpaired.fastq"

############################################
# skewer

run_and_log "$THREADS" "skewer" \
  skewer \
    -x "$ADAPTER_R1" \
    -y "$ADAPTER_R2" \
    -q 30 \
    -l 50 \
    -m pe \
    -t "$THREADS" \
    -o "${OUTDIR}/${SAMPLE}skewer_tmp" \
    "$R1" "$R2"

mv "${OUTDIR}/${SAMPLE}skewer_tmp-trimmed-pair1.fastq" \
   "${OUTDIR}/${SAMPLE}1_skewer.fastq"

mv "${OUTDIR}/${SAMPLE}skewer_tmp-trimmed-pair2.fastq" \
   "${OUTDIR}/${SAMPLE}2_skewer.fastq"

rm -f "${OUTDIR}/${SAMPLE}skewer_tmp-trimmed.log"

############################################
# CLEANUP

echo "CLEANUP"
rm -f "$ADAPTER_FASTA"
rm -f "${OUTDIR}/${SAMPLE}_adapters.fa" \
  "${OUTDIR}/${SAMPLE}1_fastp.fastq" \
  "${OUTDIR}/${SAMPLE}2_fastp.fastq" \
  "${OUTDIR}/${SAMPLE}1_cutadapt.fastq" \
  "${OUTDIR}/${SAMPLE}2_cutadapt.fastq" \
  "${OUTDIR}/${SAMPLE}1_trimmomatic.fastq" \
  "${OUTDIR}/${SAMPLE}2_trimmomatic.fastq" \
  "${OUTDIR}/${SAMPLE}1_skewer.fastq" \
  "${OUTDIR}/${SAMPLE}2_skewer.fastq"

done
done

echo "FINISHED"
echo "Resource log written to: $RESOURCE_LOG"
# Play a sound, when finished
# This could work if run from a terminal window
#printf '\a'
#But running from jupyter lab use this and have a python script at the end:
touch .job_finished

Starting fastp (2 threads)


Read1 before filtering:
total reads: 1500000
total bases: 151500000
Q20 bases: 144840043(95.604%)
Q30 bases: 135807085(89.6416%)
Q40 bases: 38853370(25.6458%)

Read2 before filtering:
total reads: 1500000
total bases: 151500000
Q20 bases: 137070524(90.4756%)
Q30 bases: 127088811(83.887%)
Q40 bases: 35425824(23.3834%)

Read1 after filtering:
total reads: 1393472
total bases: 137653790
Q20 bases: 135929599(98.7474%)
Q30 bases: 128755043(93.5354%)
Q40 bases: 37603519(27.3175%)

Read2 after filtering:
total reads: 1393472
total bases: 135656985
Q20 bases: 132727784(97.8407%)
Q30 bases: 123926081(91.3525%)
Q40 bases: 34853629(25.6925%)

Filtering result:
reads passed filter: 2786944
reads failed due to low quality: 23804
reads failed due to too many N: 298
reads failed due to too short: 187252
reads failed due to adapter dimer: 1702
reads with adapter trimmed: 52403
bases trimmed due to adapters: 1458657

Duplication rate: 0.3738%

Insert size peak (evaluated by paired-end reads): 101

JSON

Starting cutadapt (2 threads)
This is cutadapt 5.2 with Python 3.12.12
Command line parameters: -j 2 -q 30 -m 50 -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCA -A AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT -o POLL_resource_usage/COW_rep1_threads_2_1_cutadapt.fastq -p POLL_resource_usage/COW_rep1_threads_2_2_cutadapt.fastq /home/jokubas/project/data/Teaching_Data/poll_1.fastq /home/jokubas/project/data/Teaching_Data/poll_2.fastq
Processing paired-end reads on 2 cores ...

=== Summary ===

Total read pairs processed:          1,500,000
  Read 1 with adapter:                  26,296 (1.8%)
  Read 2 with adapter:                  24,969 (1.7%)

== Read fate breakdown ==
Pairs that were too short:             244,781 (16.3%)
Pairs written (passing filters):     1,255,219 (83.7%)

Total basepairs processed:   303,000,000 bp
  Read 1:   151,500,000 bp
  Read 2:   151,500,000 bp
Quality-trimmed:              35,396,161 bp (11.7%)
  Read 1:    12,475,694 bp
  Read 2:    22,920,467 bp
Total written (filtered):    

Picked up _JAVA_OPTIONS: -Xms1g -Xmx8g
TrimmomaticPE: Started with arguments:
 -threads 2 -phred33 /home/jokubas/project/data/Teaching_Data/poll_1.fastq /home/jokubas/project/data/Teaching_Data/poll_2.fastq POLL_resource_usage/COW_rep1_threads_2_1_trimmomatic.fastq POLL_resource_usage/COW_rep1_threads_2_1_trimmomatic_unpaired.fastq POLL_resource_usage/COW_rep1_threads_2_2_trimmomatic.fastq POLL_resource_usage/COW_rep1_threads_2_2_trimmomatic_unpaired.fastq ILLUMINACLIP:POLL_resource_usage/COW_rep1_threads_2__adapters.fa:2:30:10 TRAILING:30 MINLEN:50
ILLUMINACLIP: Using adapter file from current working directory: /home/jokubas/project/analysis/POLL_resource_usage/COW_rep1_threads_2__adapters.fa
Using PrefixPair: 'AGATCGGAAGAGCACACGTCTGAACTCCAGTCA' and 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Input Read Pairs: 1500000 Both Surviving: 1394779 (92.99%) Forward Only Surviving: 828

Starting skewer (2 threads)


...

Starting cutadapt (4 threads)
This is cutadapt 5.2 with Python 3.12.12
Command line parameters: -j 4 -q 30 -m 50 -a AGATCGGAAGAGCACACGTCTGAACTCCAGTCA -A AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT -o POLL_resource_usage/COW_rep2_threads_4_1_cutadapt.fastq -p POLL_resource_usage/COW_rep2_threads_4_2_cutadapt.fastq /home/jokubas/project/data/Teaching_Data/poll_1.fastq /home/jokubas/project/data/Teaching_Data/poll_2.fastq
Processing paired-end reads on 4 cores ...

=== Summary ===

Total read pairs processed:          1,500,000
  Read 1 with adapter:                  26,296 (1.8%)
  Read 2 with adapter:                  24,969 (1.7%)

== Read fate breakdown ==
Pairs that were too short:             244,781 (16.3%)
Pairs written (passing filters):     1,255,219 (83.7%)

Total basepairs processed:   303,000,000 bp
  Read 1:   151,500,000 bp
  Read 2:   151,500,000 bp
Quality-trimmed:              35,396,161 bp (11.7%)
  Read 1:    12,475,694 bp
  Read 2:    22,920,467 bp
Total written (filtered):    

Picked up _JAVA_OPTIONS: -Xms1g -Xmx8g
TrimmomaticPE: Started with arguments:
 -threads 4 -phred33 /home/jokubas/project/data/Teaching_Data/poll_1.fastq /home/jokubas/project/data/Teaching_Data/poll_2.fastq POLL_resource_usage/COW_rep2_threads_4_1_trimmomatic.fastq POLL_resource_usage/COW_rep2_threads_4_1_trimmomatic_unpaired.fastq POLL_resource_usage/COW_rep2_threads_4_2_trimmomatic.fastq POLL_resource_usage/COW_rep2_threads_4_2_trimmomatic_unpaired.fastq ILLUMINACLIP:POLL_resource_usage/COW_rep2_threads_4__adapters.fa:2:30:10 TRAILING:30 MINLEN:50
ILLUMINACLIP: Using adapter file from current working directory: /home/jokubas/project/analysis/POLL_resource_usage/COW_rep2_threads_4__adapters.fa
Using PrefixPair: 'AGATCGGAAGAGCACACGTCTGAACTCCAGTCA' and 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Input Read Pairs: 1500000 Both Surviving: 1394779 (92.99%) Forward Only Surviving: 828

Starting skewer (4 threads)


|====================>     IOPub message rate exceeded.2.70%)
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

|=================================================>| (100.00%)

.--. .-.
: .--': :.-.
`. `. : `'.' .--. .-..-..-. .--. .--.
_`, :: . `.' '_.': `; `; :' '_.': ..'
`.__.':_;:_;`.__.'`.__.__.'`.__.':_;
skewer v0.2.2 [April 4, 2016]
Parameters used:
-- 3' end adapter sequence (-x):	AGATCGGAAGAGCACACGTCTGAACTCCAGTCA
-- paired 3' end adapter sequence (-y):	AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT
-- maximum error ratio allowed (-r):	0.100
-- maximum indel error ratio allowed (-d):	0.030
-- end quality threshold (-q):		30
-- minimum read length allowed after trimming (-l):	50
-- file format (-f):		Sanger/Illumina 1.8+ FASTQ (auto detected)
-- number of concurrent threads (-t):	4
Fri Mar 13 15:48:25 2026 >> started

Fri Mar 13 15:48:32 2026 >> done (6.291s)
1500000 read pairs processed; of these:
  86448 ( 5.76%) short read pairs filtered out after trimming by size control
  15055 ( 1.00%) empty read pairs filtered out after trimming by size control
1398497 (93.23%) read pairs available; of these:
 862869 (61.70%) trimmed read pairs available after processing
 53

In [2]:
# SOUND SCRIPT
# Plays a couple of beeps, when finished with the code above
from pathlib import Path
from IPython.display import Audio, display
import numpy as np

flag = Path(".job_finished")

if flag.exists():
    rate = 44100

    # Beep sound
    t = np.linspace(0, 0.4, int(rate * 0.4), False)
    beep = 0.25 * np.sin(2 * np.pi * 440 * t)

    # Paus
    silence = np.zeros(int(rate * 0.2))

    # Pattern: beep – pause – beep
    sound = np.concatenate([beep, silence, beep, silence, beep])

    display(Audio(sound, rate=rate, autoplay=True))
    print("Finished")
    flag.unlink()   # optional: remove the flag so it only beeps once
else:
    print("Job not finished yet")

Finished
